# Workflow 2 — Guided Search

Comparaison entre la methode classique (Tache 1) et le Guided Search.

Le LLM intervient **deux fois** :
1. Premier appel : quelles transformations de patterns sont pertinentes ?
2. Deuxieme appel : quels candidats X -> Y tester en priorite ?

Ensuite l'algorithme valide seulement les candidats selectionnes.

In [1]:
pip install groq --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, json, time, re
sys.path.insert(0, '..')
import pandas as pd
import requests

from pfd_verifier import verifier_pfd
from pfd_discovery import discover

## Configuration du LLM

In [ ]:
from groq import Groq

# === CHOISIR LE MODELE ===
MODELE = "groq"
#MODELE = "deepseek"

# --- Config DeepSeek (local) ---
OLLAMA_URL = "http://localhost:11434/api/generate"
DEEPSEEK_MODEL =  "deepseek-r1:1.5b" #"deepseek-r1:8b"   

# --- Config Groq (API gratuite) ---
GROQ_API_KEY = "VOTRE_CLE_GROQ_ICI"
groq_client = Groq(api_key=GROQ_API_KEY)

# --- Fonction unique ---
def appeler_llm(prompt):
    if MODELE == "deepseek":
        resp = requests.post(OLLAMA_URL, json={
            "model": DEEPSEEK_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.1}
        }, timeout=1000)
        resp.raise_for_status()
        return resp.json()["response"]
    elif MODELE == "groq":
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        return response.choices[0].message.content

def extraire_json(texte):
    match = re.search(r'\{.*\}', texte, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            return None
    return None

print(f"Modele actif : {MODELE}")

Modele actif : groq


In [4]:
print(appeler_llm("Dis juste OK."))

C'est un peu court, n'est-ce pas ? Qu'est-ce que vous aimeriez discuter ?


## Charger le dataset et preparer le contexte

In [5]:
df = pd.read_csv('../data/pfd_validation/t2.csv')
print(f"t2.csv : {len(df)} lignes x {len(df.columns)} colonnes")
print(f"Colonnes: {list(df.columns)}")
df.head()

t2.csv : 3502 lignes x 13 colonnes
Colonnes: ['Year', 'EMPLOYER_ID', 'NAME', 'ADDRESS_1', 'ADDRESS_2', 'CITY', 'STATE', 'ZIP', 'COUNTRY', 'PHONE', 'FAX', 'CREATED_DATE', 'ACTIVE']


,Year,EMPLOYER_ID,NAME,ADDRESS_1,ADDRESS_2,CITY,STATE,ZIP,COUNTRY,PHONE,FAX,CREATED_DATE,ACTIVE
0,2011.0,4818,Lighten-Gale LLC,"39 S. LaSalle St., Ste. 808",NaN,Chicago,Il,60603,United States,312-920-1500,NaN,NaN,Y
1,2012.0,4099,Nicolay & Dart LLC,33 N. Dearborn St. Ste.2200,NaN,Chicago,IL,60602,United States,312-701-0221,312-658-0464,NaN,Y
2,2011.0,4099,Nicolay & Dart LLC,33 N. Dearborn St. Ste.2200,NaN,Chicago,IL,60602,United States,312-701-0221,312-658-0464,NaN,Y
3,2013.0,7141,Carol Ronen,6033 N. Sheridan Rd,NaN,Chicago,IL,60660,United States,773-919-4240,NaN,01/09/2013,Y
4,2011.0,4687,Thompson Coburn LLP,55 E Monroe - 40th Floor,NaN,Chicago,IL,60603,United States,312-580-2228,312-580-2201,NaN,Y


In [6]:
echantillon = df.head(5).to_string(index=False)

contexte = f"""Voici un dataset avec {len(df)} lignes et les colonnes suivantes :
{list(df.columns)}

Voici 5 lignes d'exemple :
{echantillon}
"""

## Methode classique (Tache 1) — la reference

On lance d'abord la methode classique pour avoir les resultats de reference.

In [7]:
t0 = time.time()
pfds_classique = discover(df, epsilon=0.1, min_support=5, k_max=3, verbose=True)
temps_classique = time.time() - t0
print(f"\nResultat : {len(pfds_classique)} PFDs en {temps_classique:.1f}s")


[1/4] Extraction des patterns (k_max=3, min_support=5)...
    13 colonnes, 2328 patterns

[2/4] Génération des candidats...
    23,469 candidats

[3/4] Validation (epsilon=0.1)...
    7627 PFDs valides sur 23,469 testés

[4/4] Généralisation...
    2608 règles (451 généralisées, 5019 élagées)

  Temps total : 59.46s

Resultat : 2608 PFDs en 59.5s


## Guided Search — Appel 1 : Transformations

Premier appel au LLM : on montre le schema + echantillon au llm
et on lui demande quelles transformations de patterns sont pertinentes.

In [8]:
prompt_1 = contexte + f"""
Tu es un expert en data quality et Pattern Functional Dependencies (PFDs).
Une PFD c'est une regle du type : "si le prefixe de 3 caracteres du ZIP est 606, alors CITY = Chicago".

ATTENTION : tu dois UNIQUEMENT utiliser les colonnes qui existent vraiment dans ce dataset.
Les colonnes disponibles sont EXACTEMENT : {list(df.columns)}
N'invente AUCUNE autre colonne. N'utilise QUE les noms ci-dessus, a l'identique.

Les transformations possibles sont :
- prefix(col, n) : les n premiers caracteres
- first_token(col) : le premier mot

Pour chaque colonne pertinente de ce dataset, dis-moi quelles transformations sont utiles.

Reponds UNIQUEMENT en JSON, sans texte avant ou apres :
{{
  "transformations": [
    {{"colonne": "NOM_EXACT_DE_COLONNE", "type": "prefix", "longueur": 3}},
    {{"colonne": "NOM_EXACT_DE_COLONNE", "type": "first_token"}}
  ]
}}
"""

print("Appel 1 : transformations...")
t1_start = time.time()
reponse_1 = appeler_llm(prompt_1)
t1_end = time.time()
print(f"Reponse en {t1_end - t1_start:.1f}s")
print(reponse_1[:2000])

Appel 1 : transformations...
Reponse en 0.4s
{
  "transformations": [
    {"colonne": "ZIP", "type": "prefix", "longueur": 3},
    {"colonne": "ZIP", "type": "prefix", "longueur": 5},
    {"colonne": "ADDRESS_1", "type": "first_token"},
    {"colonne": "ADDRESS_2", "type": "first_token"},
    {"colonne": "CITY", "type": "first_token"},
    {"colonne": "STATE", "type": "first_token"},
    {"colonne": "COUNTRY", "type": "first_token"},
    {"colonne": "PHONE", "type": "first_token"},
    {"colonne": "FAX", "type": "first_token"},
    {"colonne": "CREATED_DATE", "type": "first_token"}
  ]
}


In [9]:
# parser les transformations
data_1 = extraire_json(reponse_1)
if data_1 and "transformations" in data_1:
    transformations = data_1["transformations"]
    print(f"Le LLM propose {len(transformations)} transformations :")
    for t in transformations:
        if t["type"] == "prefix":
            print(f"  prefix({t['colonne']}, {t.get('longueur', 3)})")
        else:
            print(f"  first_token({t['colonne']})")
else:
    print("Erreur de parsing. Reponse brute :")
    print(reponse_1[:1000])
    transformations = []

Le LLM propose 10 transformations :
  prefix(ZIP, 3)
  prefix(ZIP, 5)
  first_token(ADDRESS_1)
  first_token(ADDRESS_2)
  first_token(CITY)
  first_token(STATE)
  first_token(COUNTRY)
  first_token(PHONE)
  first_token(FAX)
  first_token(CREATED_DATE)


## Guided Search — Appel 2 : Candidats a tester

Deuxieme appel au LLM: on lui donne les transformations
qu'il a proposees et on lui demande quels candidats X -> Y tester.

In [10]:
# on construit le deuxieme prompt en incluant les transformations du premier appel
liste_transfo = json.dumps(transformations, indent=2)

prompt_2 = contexte + f"""
Tu as propose les transformations suivantes :
{liste_transfo}

Maintenant, dis-moi quels candidats PFDs je dois tester.
Un candidat c'est : pattern(col_X) -> col_Y.

REGLE ABSOLUE : col_X et col_Y doivent OBLIGATOIREMENT faire partie de cette liste exacte :
{list(df.columns)}
Si tu utilises un nom de colonne qui n'est pas dans cette liste, le candidat sera rejete.
Ne mets PAS de colonnes comme ZIP, CITY, PHONE si elles ne sont pas dans la liste ci-dessus.

Propose entre 10 et 15 candidats qui ont un sens semantique.

Reponds UNIQUEMENT en JSON :
{{
  "candidats": [
    {{"col_X": "NOM_EXACT", "pattern_type": "prefix", "longueur": 3, "col_Y": "NOM_EXACT"}},
    {{"col_X": "NOM_EXACT", "pattern_type": "first_token", "col_Y": "NOM_EXACT"}}
  ]
}}
"""

print("Appel 2 : candidats a tester...")
t2_start = time.time()
reponse_2 = appeler_llm(prompt_2)
t2_end = time.time()
print(f"Reponse en {t2_end - t2_start:.1f}s")
print(reponse_2[:2000])

Appel 2 : candidats a tester...
Reponse en 0.9s
Voici 15 candidats PFDs qui ont un sens semantique :

```json
{
  "candidats": [
    {"col_X": "ADDRESS_1", "pattern_type": "prefix", "longueur": 3, "col_Y": "ADDRESS_1"},
    {"col_X": "ADDRESS_1", "pattern_type": "prefix", "longueur": 5, "col_Y": "ADDRESS_1"},
    {"col_X": "ADDRESS_1", "pattern_type": "first_token", "col_Y": "ADDRESS_1"},
    {"col_X": "ADDRESS_2", "pattern_type": "prefix", "longueur": 3, "col_Y": "ADDRESS_2"},
    {"col_X": "ADDRESS_2", "pattern_type": "prefix", "longueur": 5, "col_Y": "ADDRESS_2"},
    {"col_X": "ADDRESS_2", "pattern_type": "first_token", "col_Y": "ADDRESS_2"},
    {"col_X": "CITY", "pattern_type": "prefix", "longueur": 3, "col_Y": "CITY"},
    {"col_X": "CITY", "pattern_type": "prefix", "longueur": 5, "col_Y": "CITY"},
    {"col_X": "CITY", "pattern_type": "first_token", "col_Y": "CITY"},
    {"col_X": "STATE", "pattern_type": "prefix", "longueur": 2, "col_Y": "STATE"},
    {"col_X": "STATE", "patte

In [11]:
# parser les candidats
data_2 = extraire_json(reponse_2)
if data_2 and "candidats" in data_2:
    candidats_llm = data_2["candidats"]
    print(f"Le LLM propose {len(candidats_llm)} candidats a tester :")
    for c in candidats_llm:
        longueur = c.get('longueur', '')
        print(f"  {c['pattern_type']}({c['col_X']}, {longueur}) -> {c['col_Y']}")
else:
    print("Erreur de parsing. Reponse brute :")
    print(reponse_2[:1000])
    candidats_llm = []

Le LLM propose 15 candidats a tester :
  prefix(ADDRESS_1, 3) -> ADDRESS_1
  prefix(ADDRESS_1, 5) -> ADDRESS_1
  first_token(ADDRESS_1, ) -> ADDRESS_1
  prefix(ADDRESS_2, 3) -> ADDRESS_2
  prefix(ADDRESS_2, 5) -> ADDRESS_2
  first_token(ADDRESS_2, ) -> ADDRESS_2
  prefix(CITY, 3) -> CITY
  prefix(CITY, 5) -> CITY
  first_token(CITY, ) -> CITY
  prefix(STATE, 2) -> STATE
  first_token(STATE, ) -> STATE
  prefix(COUNTRY, 3) -> COUNTRY
  first_token(COUNTRY, ) -> COUNTRY
  prefix(PHONE, 3) -> PHONE
  first_token(PHONE, ) -> PHONE


## Validation des candidats selectionnes

On utilise le meme verifier_pfd que la Tache 1.
La seule difference : on ne teste QUE les candidats proposes par le LLM.

In [12]:
t_val_start = time.time()
pfds_guided = []

for c in candidats_llm:
    col_X = c["col_X"]
    col_Y = c["col_Y"]
    
    if col_X not in df.columns or col_Y not in df.columns:
        print(f"  SKIP: colonne absente")
        continue
    
    if c["pattern_type"] == "prefix":
        longueur = c.get("longueur", 3)
        prefixes = df[col_X].dropna().astype(str).str[:longueur]
        for pattern_val, count in prefixes.value_counts().head(30).items():
            if count < 5:
                continue
            mask = df[col_X].astype(str).str.startswith(str(pattern_val))
            val_counts = df.loc[mask, col_Y].value_counts()
            if val_counts.empty:
                continue
            top_Y = val_counts.index[0]
            
            res = verifier_pfd(df,
                col_X=col_X, pattern_X=pattern_val,
                col_Y=col_Y, pattern_Y=top_Y,
                epsilon=0.1,
                match_type_X="startswith", match_type_Y="exact")
            
            if res["is_valid"]:
                res.update({"col_X": col_X, "pattern_X": pattern_val,
                           "col_Y": col_Y, "pattern_Y": top_Y,
                           "match_type_X": "startswith", "match_type_Y": "exact"})
                pfds_guided.append(res)
    
    elif c["pattern_type"] == "first_token":
        tokens = df[col_X].dropna().astype(str).str.split().str[0]
        for token_val, count in tokens.value_counts().head(30).items():
            if count < 5:
                continue
            mask = df[col_X].astype(str).str.startswith(str(token_val))
            top_Y = df.loc[mask, col_Y].value_counts().index[0]
            
            res = verifier_pfd(df,
                col_X=col_X, pattern_X=token_val,
                col_Y=col_Y, pattern_Y=top_Y,
                epsilon=0.1,
                match_type_X="startswith", match_type_Y="exact")
            
            if res["is_valid"]:
                res.update({"col_X": col_X, "pattern_X": token_val,
                           "col_Y": col_Y, "pattern_Y": top_Y,
                           "match_type_X": "startswith", "match_type_Y": "exact"})
                pfds_guided.append(res)

temps_validation = time.time() - t_val_start

pfds_guided.sort(key=lambda x: -x["confidence"])
print(f"{len(pfds_guided)} PFDs valides")
print(f"Temps validation : {temps_validation:.1f}s")
print()
for p in pfds_guided[:20]:
    print(f"  {p['col_X']}['{p['pattern_X']}'] -> {p['col_Y']}={p['pattern_Y']!r}  "
          f"conf={p['confidence']:.1%}  support={p['n_matching_X']}")

190 PFDs valides
Temps validation : 0.9s

  ADDRESS_1['158'] -> ADDRESS_1='1585 Broadway'  conf=100.0%  support=38
  ADDRESS_1['1585 '] -> ADDRESS_1='1585 Broadway'  conf=100.0%  support=38
  ADDRESS_1['1 E. '] -> ADDRESS_1='1 E. Wacker Dr., Ste. 2350'  conf=100.0%  support=22
  ADDRESS_1['1585'] -> ADDRESS_1='1585 Broadway'  conf=100.0%  support=38
  ADDRESS_2['NC1'] -> ADDRESS_2='NC1-007-22-32'  conf=100.0%  support=16
  ADDRESS_2['TPN'] -> ADDRESS_2='TPN 1101'  conf=100.0%  support=8
  ADDRESS_2['21s'] -> ADDRESS_2='21st Floor'  conf=100.0%  support=6
  ADDRESS_2['DE5'] -> ADDRESS_2='DE5-001-02-07'  conf=100.0%  support=6
  ADDRESS_2['22n'] -> ADDRESS_2='22nd Floor'  conf=100.0%  support=5
  ADDRESS_2['4th'] -> ADDRESS_2='4th Floor'  conf=100.0%  support=5
  ADDRESS_2['O2N'] -> ADDRESS_2='O2N'  conf=100.0%  support=5
  ADDRESS_2['MAC'] -> ADDRESS_2='MAC P6101-022'  conf=100.0%  support=5
  ADDRESS_2['NC1-0'] -> ADDRESS_2='NC1-007-22-32'  conf=100.0%  support=16
  ADDRESS_2['TPN 1'] 

## Comparaison finale : Classique vs Guided Search

In [13]:
temps_llm_total = (t1_end - t1_start) + (t2_end - t2_start)
temps_guided_total = temps_llm_total + temps_validation

print(f"""
{'='*70}
  CLASSIQUE (Tache 1) vs GUIDED SEARCH (Workflow 2)
{'='*70}

  Critere                      Classique         Guided Search
  ------------------------------------------------------------------
  PFDs trouvees                {len(pfds_classique):<20}{len(pfds_guided)}
  Candidats testes             ~15,000             {len(candidats_llm)}
  Temps total                  {temps_classique:<20.1f}{temps_guided_total:.1f}s
    - temps LLM (2 appels)     -                   {temps_llm_total:.1f}s
    - temps validation         {temps_classique:<20.1f}{temps_validation:.1f}s
  Appels LLM                   0                   2
{'='*70}
""")


  CLASSIQUE (Tache 1) vs GUIDED SEARCH (Workflow 2)

  Critere                      Classique         Guided Search
  ------------------------------------------------------------------
  PFDs trouvees                2608                190
  Candidats testes             ~15,000             15
  Temps total                  59.5                2.1s
    - temps LLM (2 appels)     -                   1.2s
    - temps validation         59.5                0.9s
  Appels LLM                   0                   2



In [14]:
# analyse des PFDs : quelles paires col_X -> col_Y sont trouvees par chaque methode ?
paires_classique = set((p["col_X"], p["col_Y"]) for p in pfds_classique)
paires_guided = set((p["col_X"], p["col_Y"]) for p in pfds_guided)

en_commun = paires_classique & paires_guided
ratees = paires_classique - paires_guided

print(f"Paires en commun : {len(en_commun)}")
print(f"Ratees par le guided : {len(ratees)}")

if ratees:
    print(f"\nPFDs que le classique trouve mais pas le guided :")
    for p in list(ratees)[:10]:
        print(f"  {p[0]} -> {p[1]}")

print(f"\nConclusion :")
print(f"  Le classique teste ~15,000 candidats pour trouver {len(pfds_classique)} PFDs.")
print(f"  Le guided teste {len(candidats_llm)} candidats (proposes par le LLM) pour trouver {len(pfds_guided)} PFDs.")
print(f"  Le LLM permet de reduire l'espace de recherche en se concentrant sur les candidats semantiquement pertinents.")

Paires en commun : 0
Ratees par le guided : 106

PFDs que le classique trouve mais pas le guided :
  FAX -> ADDRESS_1
  ADDRESS_2 -> STATE
  PHONE -> NAME
  ADDRESS_1 -> CREATED_DATE
  NAME -> ADDRESS_1
  FAX -> STATE
  ADDRESS_1 -> ZIP
  NAME -> EMPLOYER_ID
  EMPLOYER_ID -> PHONE
  PHONE -> ACTIVE

Conclusion :
  Le classique teste ~15,000 candidats pour trouver 2608 PFDs.
  Le guided teste 15 candidats (proposes par le LLM) pour trouver 190 PFDs.
  Le LLM permet de reduire l'espace de recherche en se concentrant sur les candidats semantiquement pertinents.
